# Seq2seq와 Attention — 예제 코드 상세 주석본

**목표:** 코드 한 줄이 무엇을 계산하는지, 왜 필요한지, 텐서 크기가 어떻게 바뀌는지 따라갑니다.

[원본 강의 노트북](https://github.com/yooongZa/AIFFEL_Quest_EPA/blob/main/Seq2seq%E1%84%8B%E1%85%AA_Attention.ipynb)의 **실행 코드 8개를 순서대로 모았습니다.**
원본 강의 본문·영상·퀴즈를 복제한 전체 강의본이 아니라, **예제 코드 중심 주석본**입니다.
실행문·변수명·설정값은 유지하고, 한글 주석과 읽기 안내를 추가했습니다.

> 이 코드는 **구조와 텐서 크기를 확인하는 실습**입니다. 입력은 난수이며, 번역 데이터 전처리·학습·문장 생성은 포함하지 않습니다.
> 아래 Attention 두 예제는 앞의 디코더에 연결되지 않은 독립 실습입니다.

## 시작 전에: 실행 순서와 표기

PyTorch와 NumPy가 설치된 Python 환경에서 **위에서 아래로 모두 실행**합니다.
설치가 필요한 환경에서는 해당 커널에서 `%pip install torch numpy`를 한 번 실행하세요.
여기서는 CPU로 실행하므로 GPU가 없어도 됩니다. 난수 시드를 고정하지 않은 원본을 유지했으므로 비중의 숫자는 실행마다 달라질 수 있습니다.

| 표기 | 뜻 | 인코더·디코더 예제 | Attention 예제 |
|:--|:--|:--|:--|
| `B` | 한 번에 처리하는 문장 수(배치 크기) | 1 | 1 |
| `S` | 인코더 쪽 입력 위치 수 | 3 | 10 |
| `T` | 디코더에 넣는 토큰 수 | 3 | 반복 디코딩 없이 상태 하나만 비교 |
| `V` | 단어장에 있는 토큰 종류 수 | 30000 | 사용하지 않음 |
| `E` | 토큰 임베딩 벡터 길이 | 256 | 이미 상태 벡터가 주어짐 |
| `H` | LSTM 상태 벡터 길이 | 512 | 512 |
| `A` | Bahdanau 점수 계산용 중간 차원 | 사용하지 않음 | 100 |

`(1, 3, 256)`은 **문장 1개 × 토큰 3개 × 각 토큰의 숫자 256개**입니다.
축 번호는 0부터 시작하므로 `dim=0`은 배치, `dim=1`은 위치, `dim=2`는 특징입니다.
단, LSTM이 반환한 최종 상태의 축은 `(층 수×방향 수, 배치, 특징)`이므로 별도로 읽어야 합니다. [1]

## 1. LSTM 인코더

인코더는 토큰 번호를 벡터로 바꾸고 문장을 읽습니다. 모든 시점의 상태와 마지막 상태를 구분해서 확인합니다. [1][2]

### 예제 1 · Encoder 클래스 정의

In [1]:
# [예제 1] 인코더: 토큰 번호 → 임베딩 벡터 → 문장을 읽은 LSTM 상태
# nn에는 Embedding, LSTM, Linear 등 신경망을 구성하는 클래스가 들어 있다.
# as nn은 긴 이름 torch.nn을 nn이라는 짧은 이름으로 쓰겠다는 뜻이다.
import torch.nn as nn

# nn.Module을 상속해 PyTorch 모델 클래스를 만든다.
# class는 설계도이며, 아래 셀의 Encoder(...)에서 실제 모델 객체를 만든다.
class Encoder(nn.Module):

    # __init__은 모델 객체를 만들 때 실행되는 초기 설정이다.
    # self는 지금 만들고 있는 모델 객체를 가리킨다.
    # input_dim: 입력 토큰 종류 수(V), emb_dim: 토큰 벡터 길이(E),
    # hidden_dim: LSTM의 은닉 상태 벡터 길이(H). 셋은 서로 다른 개념이다.
    def __init__(self, input_dim, emb_dim, hidden_dim):

        # 부모 클래스 nn.Module을 초기화한다.
        # self에 저장한 레이어와 학습할 파라미터를 PyTorch가 관리할 수 있게 한다.
        super().__init__()

        # 단어장 번호를 E개의 실수로 된 벡터로 바꾸는 조회표를 만든다.
        # 예를 들어 nn.Embedding(30000, 256)은 (30000, 256) 크기의 표이다.
        # 토큰 번호 7이 들어오면 7번 행의 벡터를 꺼낸다. 원-핫 벡터를 만들지는 않는다.
        # 현재 표는 무작위 초기값이다. 의미 있는 벡터가 되려면 학습이 필요하다.
        self.embedding = nn.Embedding(input_dim, emb_dim)

        # 한 시점에 E차원 임베딩을 받고 H차원 은닉 상태를 만드는 LSTM이다.
        # batch_first=True: 입력과 시퀀스 출력의 축 순서를 (B, S, 특징 차원)으로 둔다.
        # 설정을 생략한 num_layers=1, bidirectional=False이므로 1층·단방향이다.
        # 주의: batch_first는 최종 hidden/cell의 축 순서를 바꾸지 않는다.
        self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    # encoder(src)로 모델을 호출하면 이 forward가 계산을 수행한다.
    # src는 아직 벡터가 아니라 정수 토큰 번호 묶음이다. 크기: (B, S).
    def forward(self, src):

        # .size()와 .shape는 여기서 모두 텐서의 각 축 크기를 확인하는 용도다.
        # 실습 입력 (1, 3)은 문장 1개에 토큰이 3개 있다는 뜻이다.
        print("입력 Shape:", src.size())

        # 각 토큰 번호를 256차원 벡터로 바꾼다.
        # (B, S) → (B, S, E), 실습에서는 (1, 3) → (1, 3, 256).
        # 문장 수와 토큰 수는 그대로이고, 토큰마다 벡터 축이 추가된다.
        embedded = self.embedding(src)
        print("Embedding Layer를 거친 Shape:", embedded.size())

        # LSTM이 문장을 앞에서부터 읽으며 hidden과 cell을 갱신한다.
        # 초기 (hidden, cell)을 직접 넘기지 않았으므로 둘 다 0에서 시작한다.
        # 중요: 왼쪽 변수 이름은 h_0, c_0이지만, 여기서 받는 값은 초기 상태가 아니다.
        # PyTorch가 반환한 최종 상태 h_n, c_n을 원본에서 이렇게 이름 붙인 것이다.
        # outputs: 모든 시점의 은닉 상태 → (B, S, H) = (1, 3, 512).
        # h_0: 마지막 시점의 은닉 상태 → (층 수×방향 수, B, H) = (1, 1, 512).
        # c_0: 마지막 시점의 셀 상태 → 이 예제에서는 역시 (1, 1, 512).
        # h는 바깥으로 내보내는 상태, c는 LSTM 내부에서 이어 가는 기억 상태이다.
        # outputs는 최상위 층의 h들을 담으며, 모든 시점의 c를 담는 것은 아니다.
        outputs, (h_0, c_0) = self.rnn(embedded)
        print("LSTM Layer의 Output Shape:", outputs.size())
        print("LSTM Layer의 Hidden State Shape:", h_0.size())
        print("LSTM Layer의 Cell State Shape:", c_0.size())

        # 서로 역할이 다른 세 결과를 호출한 곳으로 돌려준다.
        # 다음 예제에서 outputs는 sample_output, 최종 두 상태는 hidden/cell로 받는다.
        return outputs, h_0, c_0

# 클래스 정의 셀이 끝났음을 알리는 출력이다. 아직 입력을 처리한 것은 아니다.
print("슝~")

슝~


### 예제 2 · 실습 크기 설정

`vocab_size`, `emb_size`, `lstm_size`, `sample_seq_len`은 각각 다른 축의 크기입니다.

In [2]:
# [예제 2] 인코더·디코더 실습에 사용할 크기 설정
# V: 단어장의 토큰 종류 수이다. 문장 길이가 30,000이라는 뜻은 아니다.
vocab_size = 30000

# E: 토큰 하나를 표현하는 실수 벡터의 길이이다.
emb_size = 256

# H: LSTM의 은닉 상태와 셀 상태의 벡터 길이이다.
# 이 예제에서는 별도 projection을 쓰지 않으므로 둘의 크기가 같다.
lstm_size = 512

# B: 한 번에 묶어서 처리할 문장 수이다. 여기서는 문장 1개를 쓴다.
batch_size = 1

# S: 실습 입력 문장의 토큰 수이다. 디코더 예제에서도 편의상 3을 쓴다.
# 실제 Seq2seq에서 입력 문장 길이와 출력 문장 길이는 달라도 된다.
sample_seq_len = 3

# 설정값을 출력한다. {0} 자리에 .format(...)의 첫 번째 인자가 들어간다.
# 출력문 마지막의 \n은 줄바꿈 문자이다. 이 셀에는 학습 연산이 없다.
print("Vocab Size: {0}".format(vocab_size))
print("Embedding Size: {0}".format(emb_size))
print("LSTM Size: {0}".format(lstm_size))
print("Batch Size: {0}".format(batch_size))
print("Sample Sequence Length: {0}\n".format(sample_seq_len))

Vocab Size: 30000
Embedding Size: 256
LSTM Size: 512
Batch Size: 1
Sample Sequence Length: 3



### 예제 3 · 인코더 실행

`encoder(...)`를 호출하면 클래스의 `forward(...)`가 실행됩니다.

In [3]:
# [예제 3] 정수 토큰 번호를 만들어 인코더에 통과시키기
# torch는 텐서를 만들고 연산하는 PyTorch의 기본 모듈이다.
import torch

# 앞서 정의한 설계도로 인코더 객체를 만든다.
# 입력 토큰 종류 30000개, 임베딩 256차원, LSTM 상태 512차원이다.
# 레이어를 생성하는 단계이며 아직 학습된 번역기는 아니다.
encoder = Encoder(vocab_size, emb_size, lstm_size)

# 0 이상 vocab_size 미만의 정수를 (B, S) 크기로 뽑는다.
# 실습에서는 0~29999 사이의 정수 3개가 [[12, 81, 5]] 같은 형태로 생긴다.
# 이 숫자는 예시일 뿐이며 실행마다 달라질 수 있다. 실제 문장을 토큰화하지 않는다.
# 기본 정수 자료형 int64는 Embedding의 토큰 인덱스로 사용할 수 있다.
sample_input = torch.randint(0, vocab_size, (batch_size, sample_seq_len))

# encoder(...) 호출이 Encoder.forward(...)로 연결된다.
# 반환값 세 개를 왼쪽 변수 세 개에 순서대로 나누어 담는다.
# sample_output: (1, 3, 512), hidden: (1, 1, 512), cell: (1, 1, 512).
# 현재 hidden/cell은 인코더 최종 상태이며, 뒤에서 디코더의 초기 상태로 쓴다.
# 지금은 순전파만 수행한다. loss, backward, optimizer 단계는 없다.
sample_output, hidden, cell = encoder(sample_input)

입력 Shape: torch.Size([1, 3])
Embedding Layer를 거친 Shape: torch.Size([1, 3, 256])
LSTM Layer의 Output Shape: torch.Size([1, 3, 512])
LSTM Layer의 Hidden State Shape: torch.Size([1, 1, 512])
LSTM Layer의 Cell State Shape: torch.Size([1, 1, 512])


### 인코더에서 확인할 결과

| 변수 | 실제 크기 | 담고 있는 것 |
|:--|:--|:--|
| `sample_input` | `(1, 3)` | 정수 토큰 번호 3개 |
| `embedded` | `(1, 3, 256)` | 각 토큰의 임베딩 벡터 |
| `sample_output` | `(1, 3, 512)` | 세 시점의 은닉 상태 |
| `hidden` | `(1, 1, 512)` | 인코더 최종 은닉 상태 |
| `cell` | `(1, 1, 512)` | 인코더 최종 셀 상태 |

**혼동 주의:** 원본 클래스 안의 `h_0`, `c_0`는 이름과 달리 **반환된 최종 상태**입니다. 이름은 원본 그대로 두고 주석으로 구분했습니다. [1]

## 2. LSTM 디코더와 고정 context

이 구현은 인코더의 최종 `hidden/cell`을 디코더 초기 상태로 전달합니다.
동시에 최종 `hidden`을 각 시점 입력에 붙입니다.
**같은 context를 넣는다고 디코더의 hidden/cell까지 같은 값으로 고정되는 것은 아닙니다.** [1]

### 예제 4 · Decoder 클래스 정의

핵심은 `256 + 512 = 768`과 `512 → 30000`이 각각 왜 필요한지 이해하는 것입니다. [3]

In [9]:
# [예제 4] 디코더: 입력 토큰 + 고정 context → 다음 토큰 후보별 점수
# 이 예제는 인코더의 최종 상태로 디코더를 시작하고,
# 추가로 동일한 context 벡터를 디코더의 각 시점 입력에 붙이는 구조이다.
# Seq2seq의 모든 구현이 반드시 이 방식으로 context를 붙이는 것은 아니다.
class Decoder(nn.Module):

    # vocab_size는 디코더가 사용할 출력 언어의 토큰 종류 수이다.
    # 여기서는 실습 편의상 인코더와 같은 수를 쓴다. 실제 번역에서는 달라도 된다.
    def __init__(self, vocab_size, embedding_dim, hidden_dim):

        # super().__init__()과 같은 목적의 부모 클래스 초기화이다.
        super(Decoder, self).__init__()

        # 디코더용 임베딩 표를 새로 만든다.
        # 크기가 인코더 표와 같더라도 같은 파라미터를 공유하는 표는 아니다.
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 각 시점에 [토큰 임베딩 E개 + context H개]를 함께 입력한다.
        # 그래서 입력 특징 수는 E+H=256+512=768이다.
        # 반환할 LSTM 은닉 상태의 길이는 H=512로 설정한다.
        self.lstm = nn.LSTM(embedding_dim + hidden_dim, hidden_dim, batch_first=True)

        # fc는 fully connected(완전연결) 레이어라는 뜻으로 쓴 이름이다.
        # 512개 특징을 단어장 후보 30000개 각각의 점수로 바꾼다.
        # Linear는 마지막 축만 바꾸고 앞의 배치·시점 축은 유지한다.
        self.fc = nn.Linear(hidden_dim, vocab_size)

    # x: 디코더 입력 토큰 번호 (B, T).
    # hidden/cell: 디코더가 계산을 시작할 상태. 첫 호출에서는 인코더 최종 상태.
    # context: 각 시점에 덧붙일 정보 (B, T, H). 여기서는 모든 시점에 같은 값.
    # T는 디코더 입력 길이이다. 이 실습에서는 T=3으로 둔다.
    def forward(self, x, hidden, cell, context):
        print("입력 Shape:", x.size())

        # 입력 토큰 번호를 디코더의 임베딩 표에서 벡터로 바꾼다.
        # (B, T) → (B, T, E), 실습에서는 (1, 3) → (1, 3, 256).
        embedded = self.embedding(x)
        print("Embedding Layer를 거친 Shape:", embedded.size())

        # cat은 concatenate의 줄임말로, 원소별 덧셈이 아니라 이어 붙이기이다.
        # dim=2는 (배치, 시점, 특징) 중 마지막 특징 축을 가리킨다.
        # (1, 3, 256)과 (1, 3, 512)를 붙이면 (1, 3, 768)이 된다.
        # 비유: [a, b]와 [c, d, e]를 붙여 [a, b, c, d, e]를 만드는 것.
        # 따라서 이어 붙이지 않는 축인 배치 수와 시점 수는 서로 같아야 한다.
        # 다음 출력문의 '더해진'은 이 코드에서 '이어 붙인'이라는 뜻으로 읽는다.
        embedded = torch.cat((embedded, context), dim=2)
        print("Context Vector가 더해진 Shape:", embedded.size())

        # 이번에는 초기 (hidden, cell)을 LSTM에 명시적으로 전달한다.
        # 입력 (B, T, E+H) → 시점별 출력 (B, T, H), 실습에서는 (1, 3, 512).
        # 함수를 한 번 호출해도 LSTM 내부의 시점 계산은 순차적으로 이어진다.
        # 고정된 것은 추가 입력 context이며, 디코더의 hidden/cell은 계속 갱신된다.
        # 왼쪽 hidden/cell 변수에는 디코더가 처리를 끝낸 뒤의 최종 상태가 들어간다.
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        print("LSTM Layer의 Output Shape:", output.size())

        # 각 시점의 512차원 상태를 30000개 토큰 후보의 점수로 바꾼다.
        # (B, T, H) → (B, T, V), 실습에서는 (1, 3, 30000).
        # 이 결과는 확률이 아니라 정규화하지 않은 점수(logits)이다.
        # 아직 단어를 선택하거나 실제 문자열로 변환한 결과도 아니다.
        # 나중에 CrossEntropyLoss로 학습할 때는 사전 softmax를 적용하지 않는다.
        # 그때는 (B*T, V)처럼 손실 함수가 기대하는 클래스 축 형태도 맞춰야 한다.
        output = self.fc(output)
        print("Decoder 최종 Output Shape:", output.size())

        # 모든 시점의 후보 점수와 디코더 최종 상태 두 개를 돌려준다.
        return output, hidden, cell

### 예제 5 · 설정값 확인

In [5]:
# [예제 5] 디코더 실습 전에 설정값 다시 확인하기
# 앞에서 정의한 변수들을 그대로 출력한다. 설정을 바꾸거나 모델을 학습하지 않는다.
# 이 셀만 먼저 실행하면 변수가 없으므로 위쪽 셀부터 순서대로 실행한다.
print("Vocab Size: {0}".format(vocab_size))
print("Embedding Size: {0}".format(emb_size))
print("LSTM Size: {0}".format(lstm_size))
print("Batch Size: {0}".format(batch_size))
print("Sample Sequence Length: {0}\n".format(sample_seq_len))

Vocab Size: 30000
Embedding Size: 256
LSTM Size: 512
Batch Size: 1
Sample Sequence Length: 3



### 예제 6 · 인코더와 디코더 연결

`transpose`는 축 순서를 바꾸고, `expand`는 길이 1인 축을 늘려 같은 값을 여러 위치에서 참조하게 합니다. [4]

In [6]:
# [예제 6] 인코더 상태를 디코더에 전달하기
# 디코더 입력도 실제 번역문이 아니라 모양을 확인하기 위한 무작위 토큰 번호다.
# 실제 학습에서 정답을 한 칸 밀어 넣는 처리나, 추론에서 이전 예측을 넣는 처리는 없다.
# 크기: (B, T) = (1, 3).
decoder_input = torch.randint(0, vocab_size, (batch_size, sample_seq_len))

# 디코더 객체를 생성한다. 위에서 만든 인코더와는 별개의 모델이다.
decoder = Decoder(vocab_size, emb_size, lstm_size)

# 인코더 최종 상태를 디코더 매 스텝에 같은 값으로 붙임
# 현재 hidden은 인코더가 반환한 최종 은닉 상태 (1, B, H)이다.
# 1) transpose(0, 1): 0번 축과 1번 축을 맞바꾼다. (1, B, H) → (B, 1, H).
#    B=1이면 숫자 모양은 (1, 1, 512)로 같지만 축의 의미가 바뀐다.
# 2) expand(-1, T, -1): 가운데 길이 1인 축을 T로 늘려 보이게 한다.
#    -1은 해당 축의 크기를 그대로 유지하라는 뜻이다.
#    (B, 1, H) → (B, T, H), 실습에서는 (1, 1, 512) → (1, 3, 512).
# 결과적으로 모든 디코더 시점이 동일한 인코더 최종 h를 받는다.
# expand는 데이터를 여러 벌 복사하는 대신 같은 저장공간을 참조하는 뷰를 만든다.
# 주의: 이 형태는 현재의 1층·단방향 LSTM을 전제로 한다.
# 다층·양방향 모델로 바꾸면 어느 층/방향을 쓸지와 상태 연결 방법을 다시 정해야 한다.
context = hidden.transpose(0, 1).expand(-1, sample_seq_len, -1)

# 오른쪽의 hidden/cell(인코더 최종 상태)을 디코더 초기 상태로 전달한다.
# 반환된 dec_output의 크기는 (1, 3, 30000)이다.
# 주의: 대입이 끝나면 hidden/cell은 이제 디코더 최종 상태로 덮어써진다.
# 이 셀만 다시 실행하면 인코더 상태 대신 이전 디코더 상태를 쓰게 된다.
# 같은 흐름을 다시 확인하려면 인코더 실행 셀부터 순서대로 재실행한다.
# 여기까지의 디코더는 아래 Attention 클래스를 사용하지 않는 고정-context 예제이다.
dec_output, hidden, cell = decoder(decoder_input, hidden, cell, context)

입력 Shape: torch.Size([1, 3])
Embedding Layer를 거친 Shape: torch.Size([1, 3, 256])
Context Vector가 더해진 Shape: torch.Size([1, 3, 768])
LSTM Layer의 Output Shape: torch.Size([1, 3, 512])
Decoder 최종 Output Shape: torch.Size([1, 3, 30000])


### 디코더에서 확인할 결과

| 단계 | 실제 크기 | 의미 |
|:--|:--|:--|
| 디코더 입력 | `(1, 3)` | 입력 토큰 번호 |
| 임베딩 | `(1, 3, 256)` | 토큰당 256개 특징 |
| 고정 context | `(1, 3, 512)` | 각 시점에 동일한 인코더 정보 |
| 이어 붙이기 `cat` | `(1, 3, 768)` | 256개와 512개 특징 결합 |
| LSTM 출력 | `(1, 3, 512)` | 디코더의 시점별 상태 |
| 최종 `dec_output` | `(1, 3, 30000)` | 시점별 30000개 토큰 후보의 점수 |

`dec_output`은 **문자열도 확률도 아닌 logits**입니다. `CrossEntropyLoss`로 학습할 때는 미리 softmax를 적용하지 않습니다. [5]
실제 번역기로 만들려면 데이터, 정답과 한 칸 어긋난 디코더 입력, 손실 함수, 파라미터 갱신, 시작·종료 토큰과 생성 반복 과정 등이 더 필요합니다. [9]

## 3. Bahdanau Attention

**질문:** 지금 디코더가 다음 토큰을 만들려면, 인코더의 어느 위치를 얼마나 참고해야 할까요?

이 예제는 `상태 변환 → 더하기·tanh → 위치별 점수 → softmax → 원본 상태의 가중합` 순서입니다.
마지막 하나의 인코더 상태만 쓰는 대신, 모든 위치의 상태를 참고합니다. [6][9]

### 예제 7 · 점수, 비중, context를 차례로 계산

In [10]:
# [예제 7] Bahdanau Attention: 두 상태를 변환해 더한 뒤 중요도 점수 계산
# 이 셀은 앞의 인코더·디코더와 연결되지 않은 독립적인 Attention 실습이다.
# 아래에서 사용할 가상 인코더/디코더 상태를 별도로 만든다.
import torch
import torch.nn as nn

# F에는 softmax처럼 함수 형태로 호출하는 신경망 연산이 들어 있다.
import torch.nn.functional as F

# 핵심 흐름: 상태 변환 → 점수(score) → 비중(weights) → 가중합(context).
# 학습하는 것은 Linear의 파라미터이며, Attention 비중은 입력마다 계산된다.
class BahdanauAttention(nn.Module):

    # hidden_dim=H=512: 들어오는 인코더/디코더 상태 벡터의 길이.
    # units=A=100: 두 상태를 비교할 때 사용할 중간 특징 벡터의 길이.
    # A는 입력 토큰 수나 최종 context 길이가 아니다.
    # 이 구현은 인코더와 디코더의 상태 차원이 같은 경우를 가정한다.
    def __init__(self, hidden_dim, units):

        # nn.Module의 기본 기능을 초기화한 뒤 비교용 레이어들을 등록한다.
        super(BahdanauAttention, self).__init__()

        # 디코더 상태를 512차원에서 비교용 100차원으로 변환한다.
        # W_decoder라는 이름이지만 실제 객체는 가중치와 편향을 가진 Linear 레이어다.
        self.W_decoder = nn.Linear(hidden_dim, units)  # Decoder hidden state -> units

        # 인코더의 각 시점 상태도 같은 길이의 100차원으로 변환한다.
        # 디코더 변환과 출력 길이는 같지만, 학습하는 파라미터는 별개이다.
        self.W_encoder = nn.Linear(hidden_dim, units)  # Encoder hidden state -> units

        # 합쳐진 100개 특징을 입력 위치마다 스칼라 점수 1개로 바꾼다.
        # 이 단계의 출력은 아직 비중(weight)이 아니라 정규화 전 점수(score)이다.
        # 점수에는 음수가 나올 수 있고, 모든 위치의 합이 1일 필요도 없다.
        self.W_combine = nn.Linear(units, 1)           # 100차원 특징 -> softmax 이전의 스칼라 점수


    # H_encoder: 입력 문장의 모든 위치에 대한 상태 (B, S, H) = (1, 10, 512).
    # H_decoder: 참고할 디코더 시점의 상태 한 개 (B, H) = (1, 512).
    # 원래 Bahdanau 방식에서는 직전 디코더 상태를 비교에 사용한다.
    # 여기서는 디코더 반복 과정이 없으므로 해당 상태를 인자로 전달받기만 한다.
    def forward(self, H_encoder, H_decoder):
        print("[ H_encoder ] Shape:", H_encoder.shape)          # (batch, seq_len, hidden_dim)

        # 각 입력 위치의 512차원 상태를 100차원 특징으로 바꾼다.
        # (1, 10, 512) → (1, 10, 100). 10개 위치 각각에 동일한 Linear를 적용한다.
        # 토큰 위치 축 10을 줄이는 것이 아니라, 마지막 특징 축 512를 바꾼다.
        W_enc = self.W_encoder(H_encoder)                       # (batch, seq_len, units)
        print("[ W_encoder X H_encoder ] Shape:", W_enc.shape)
        print("\n[ H_decoder ] Shape:", H_decoder.shape)        # (batch, hidden_dim)

        # unsqueeze(1)은 1번 위치에 크기 1인 축을 추가한다.
        # (1, 512) → (1, 1, 512) → Linear → (1, 1, 100).
        # 가운데 1은 이번에 비교하는 디코더 상태가 하나라는 뜻이다.
        # 나중에 인코더의 10개 위치 모두와 비교할 수 있도록 축을 맞춘다.
        W_dec = self.W_decoder(H_decoder.unsqueeze(1))          # (batch, 1, units)
        print("[ W_decoder X H_decoder ] Shape:", W_dec.shape)

        # 한 줄을 세 단계로 나누어 읽는다.
        # 1) W_dec + W_enc: (B, 1, A) + (B, S, A) → (B, S, A).
        #    broadcasting 덕분에 디코더 특징 하나를 모든 인코더 위치에 맞춰 더한다.
        #    여기서 +는 원소별 덧셈이다. 앞의 torch.cat과는 다른 연산이다.
        # 2) tanh: 합쳐진 값에 비선형 변환을 적용한다. 크기는 (1, 10, 100) 그대로다.
        #    단순한 선형 변환만 이어지는 것이 아니라 더 유연한 점수 함수를 만든다.
        # 3) W_combine: 100개 특징을 점수 1개로 바꾼다. 결과는 (1, 10, 1).
        # 즉, 입력 위치 10곳이 각각 '지금 얼마나 참고할지'에 대한 점수 하나를 얻는다.
        # 이는 학습되는 적합도 점수이지, 단어 뜻의 유사도를 직접 측정한 값은 아니다.
        score = self.W_combine(torch.tanh(W_dec + W_enc))       # (batch, seq_len, 1)
        print("[ Score_alignment ] Shape:", score.shape)

        # 입력 위치 축 S(dim=1)를 따라 점수를 비중으로 바꾼다.
        # 각 문장에서 10개 위치의 비중을 합하면 부동소수점 오차 범위에서 1이 된다.
        # 이 softmax는 '입력의 어느 위치를 볼까?'를 정하는 것이지,
        # 단어장 30000개 중 다음 단어를 고르는 softmax가 아니다.
        # 주의: 마지막 축 크기가 1이므로 여기서 dim=-1을 쓰면 각 값이 1이 된다.
        # 그러면 10개 위치 사이의 비중을 비교하는 Attention이 되지 않는다.
        attention_weights = F.softmax(score, dim=1)             # (batch, seq_len, 1)

        # 출력할 때만 모양과 자료형을 바꾼다.
        # squeeze(-1): 마지막 크기 1인 축 제거 → (1, 10).
        # detach(): 출력용 텐서를 자동미분 그래프에서 분리한다.
        # numpy(): CPU 텐서를 NumPy 배열로 바꾸어 숫자를 표시한다.
        # 원래 attention_weights 변수에는 다시 대입하지 않았으므로,
        # 아래 context 계산의 자동미분 연결을 끊는 것은 아니다.
        # GPU로 옮겨 실행한다면 표시 부분은 .detach().cpu().numpy()가 필요하다.
        print("\n최종 Weight:\n", attention_weights.squeeze(-1).detach().numpy())

        # 가중합은 사영(W_encoder)하기 전의 원본 인코더 상태에 적용한다
        # 각 위치의 비중을 해당 위치의 원본 512차원 상태 전체에 곱한다.
        # (B, S, 1) * (B, S, H) → broadcasting으로 (B, S, H).
        # 그다음 sum(dim=1)으로 위치 축을 더한다. (B, S, H) → (B, H).
        # 개념 예: 위치가 3개이고 비중이 [0.7, 0.2, 0.1]이면
        # context = 0.7*h1 + 0.2*h2 + 0.1*h3 이다. 실제 실습에서는 10개를 더한다.
        # 중요: 점수 계산용 W_enc(100차원)가 아니라 H_encoder(512차원)를 합친다.
        # 따라서 최종 context는 100차원이 아닌 (1, 512)이다.
        context_vector = torch.sum(attention_weights * H_encoder, dim=1)   # (batch, hidden_dim)
        print("\n[ Context Vector ] Shape:", context_vector.shape)

        # 반환 크기: context (B, H), attention_weights (B, S, 1).
        # context는 디코더가 참고할 정보이고, weights는 각 입력 위치의 비중이다.
        # 디코더 상태가 달라지면 비중과 context 값도 달라질 수 있다.
        # context의 '벡터 길이'는 여전히 H로 고정되어 있다.
        return context_vector, attention_weights

# 설정
# 입력 상태 벡터 길이 H를 지정한다. 앞의 실습과 값이 같아도 새 가상 데이터다.
hidden_dim = 512
# 비교용 중간 차원 A를 지정한다. 여기서 units로 전달되는 값이다.
W_size = 100
print(f"Hidden State를 {W_size}차원으로 Mapping\n")

# 모델 생성
# 512차원 상태를 받아 100차원 공간에서 비교하는 모델을 만든다.
# 학습하지 않았으므로 현재 Linear의 파라미터는 초기값 상태다.
attention = BahdanauAttention(hidden_dim, W_size)

# 입력 데이터 (배치 크기 = 1)
# torch.rand는 0 이상 1 미만의 실수 난수를 만든다.
# 인코더가 문장 1개의 10개 위치에서 512차원 상태를 냈다고 가정한 테스트 데이터다.
# 앞의 sample_output을 가져온 값이 아니며, 실제 문장의 의미는 담겨 있지 않다.
enc_state = torch.rand((1, 10, hidden_dim))  # (batch, seq_len, hidden_dim)

# 디코더 상태 한 개도 (1, 512)의 실수 난수로 만든다.
# 이 Attention의 인자는 LSTM 반환 상태 (1, B, H)와 달리 (B, H)이다.
# 실제 1층 LSTM 상태를 연결할 때는 맨 앞 층 축을 골라 제거해야 한다.
dec_state = torch.rand((1, hidden_dim))      # (batch, hidden_dim)

# 실행
# 모델을 호출해 score, weights, context를 계산하고 크기와 비중을 출력한다.
# _는 반환값을 이후에 쓰지 않겠다는 관례적 변수 이름이다.
# 실제로는 (context_vector, attention_weights) 튜플이 _에 저장된다.
# 입력과 파라미터가 난수이므로 비중이 큰 위치에 의미가 있다고 해석하면 안 된다.
# 원본에 난수 시드 설정이 없어서 재실행하면 구체적인 숫자가 달라질 수 있다.
_ = attention(enc_state, dec_state)

Hidden State를 100차원으로 Mapping

[ H_encoder ] Shape: torch.Size([1, 10, 512])
[ W_encoder X H_encoder ] Shape: torch.Size([1, 10, 100])

[ H_decoder ] Shape: torch.Size([1, 512])
[ W_decoder X H_decoder ] Shape: torch.Size([1, 1, 100])
[ Score_alignment ] Shape: torch.Size([1, 10, 1])

최종 Weight:
 [[0.10217875 0.08684491 0.10029238 0.11223239 0.10623264 0.10029636
  0.09891682 0.10192978 0.10046417 0.0906117 ]]

[ Context Vector ] Shape: torch.Size([1, 512])


### Bahdanau의 크기 변화

| 계산 | 실제 크기 |
|:--|:--|
| 원본 인코더 상태 `H_encoder` | `(1, 10, 512)` |
| 비교용 인코더 특징 `W_enc` | `(1, 10, 100)` |
| 비교용 디코더 특징 `W_dec` | `(1, 1, 100)` |
| 덧셈·tanh 결과 | `(1, 10, 100)` |
| 위치별 점수 `score` | `(1, 10, 1)` |
| 위치별 비중 `attention_weights` | `(1, 10, 1)` |
| 가중합 `context_vector` | `(1, 512)` |

**100차원은 비교용이며, 최종 context는 512차원입니다.**
같은 인코더 상태를 두고도 전달한 디코더 상태에 따라 비중과 context 값이 달라질 수 있습니다.
벡터의 길이가 달라진다는 뜻은 아닙니다. [9]

## 4. Luong Attention — general 점수 방식

이번에는 `인코더 상태 변환 → 디코더 상태와 내적 → softmax → 원본 상태의 가중합`입니다.
`bmm` 두 번이 각각 **점수 계산**과 **가중합 계산**을 담당한다는 점에 집중하세요. [7][10]

### 예제 8 · 행렬곱으로 점수와 가중합 계산

In [8]:
# [예제 8] Luong Attention의 general 방식: 변환한 인코더 상태와 디코더 상태의 내적
# 앞 셀에서 import한 torch, nn, F를 사용한다. 위에서부터 순서대로 실행한다.
# Bahdanau처럼 둘을 더한 뒤 tanh를 거치지 않고, 행렬곱으로 비교 점수를 만든다.
class LuongAttention(nn.Module):
    # 여기서 units는 상태 벡터 길이 H=512이다.
    # 앞의 Bahdanau 예제에서 units=A=100이었던 것과 이름만 같고 역할이 다르다.
    # 이 구현은 인코더와 디코더 상태의 마지막 차원이 같아야 한다.
    def __init__(self, units):
        # nn.Module의 기본 기능을 초기화한다.
        super(LuongAttention, self).__init__()
        # 인코더 상태를 512차원에서 다시 512차원으로 변환한다.
        # 차원 수가 그대로라고 해서 값도 그대로인 것은 아니다.
        # Linear는 새로운 특징 조합을 학습한다. 별도 100차원 중간 공간을 쓰지 않을 뿐이다.
        # 논문의 general 점수는 h_dec의 전치 × W × h_enc 형태이다.
        # 이 예제의 nn.Linear는 기본 설정상 bias(편향)도 포함한다.
        self.W_combine = nn.Linear(units, units)  # Encoder hidden state 변환

    # H_encoder: (B, S, H) = (1, 10, 512), H_decoder: (B, H) = (1, 512).
    # Luong 방식에서는 보통 현재 디코더 상태로 Attention을 계산한다.
    # 이 독립 실습은 전달받은 상태를 비교할 뿐, 디코더 자체를 갱신하지 않는다.
    def forward(self, H_encoder, H_decoder):
        print("[ H_encoder ] Shape:", H_encoder.shape)  # (batch, seq_len, hidden_dim)

        # 모든 입력 위치에 같은 Linear를 적용한다.
        # (1, 10, 512) → (1, 10, 512). 모양은 같지만 상태 값은 변환된다.
        # 출력문의 W_encoder는 설명용 문자열이고, 실제 레이어 이름은 W_combine이다.
        WH = self.W_combine(H_encoder)  # (batch, seq_len, hidden_dim)
        print("[ W_encoder X H_encoder ] Shape:", WH.shape)

        # 디코더 상태에 길이 1인 축을 추가한다. (B, H) → (B, 1, H).
        # 다음 행렬곱에 필요한 3차원 텐서로 만드는 과정이다.
        H_decoder = H_decoder.unsqueeze(1)  # (batch, 1, hidden_dim)

        # transpose(1, 2): (B, 1, H) → (B, H, 1).
        # bmm은 배치별 행렬곱이다. 두 입력의 배치 크기는 같아야 한다.
        # (B, S, H) @ (B, H, 1) → (B, S, 1).
        # 실습에서는 (1, 10, 512) @ (1, 512, 1) → (1, 10, 1).
        # 안쪽 크기 512가 같아야 곱할 수 있고, 그 축을 따라 곱한 값들이 더해진다.
        # 결국 변환된 인코더 상태 10개 각각과 디코더 상태 하나의 내적을 구한다.
        # 원소별 곱셈 *와 달리, 행렬곱 bmm에는 곱한 값을 합치는 과정이 포함된다.
        # alignment는 비중이 아니라 아직 정규화하지 않은 위치별 점수다.
        alignment = torch.bmm(WH, H_decoder.transpose(1, 2))  # (batch, seq_len, 1)
        print("[ Score_alignment ] Shape:", alignment.shape)

        # 입력 위치 10곳의 점수를 비중으로 바꾼다.
        # 각 문장에서 위치 축 dim=1을 따라 합하면 약 1이다. 크기는 (1, 10, 1).
        attention_weights = F.softmax(alignment, dim=1)  # (batch, seq_len, 1)
        # 표시용으로만 (1, 10)으로 줄이고 자동미분에서 분리해 NumPy로 바꾼다.
        # 원본 attention_weights는 그대로 남아 아래 가중합의 자동미분에 사용된다.
        # 현재는 CPU 실습이다. GPU 사용 시에는 .detach().cpu().numpy()로 표시한다.
        print("\n최종 Weight:\n", attention_weights.squeeze(-1).detach().numpy())

        # 이번에는 squeeze 결과를 변수에 다시 저장한다.
        # (B, S, 1) → (B, S). 따라서 반환할 weights 크기도 Bahdanau 예제와 다르다.
        # squeeze는 크기 1인 축을 없애는 것이지, 값들을 합하거나 평균내는 것이 아니다.
        attention_weights = attention_weights.squeeze(-1)  # (batch, seq_len)
        # 비중에 축 하나를 더해 (B, 1, S)로 만든 뒤 원본 상태와 행렬곱한다.
        # (B, 1, S) @ (B, S, H) → (B, 1, H).
        # 실습에서는 (1, 1, 10) @ (1, 10, 512) → (1, 1, 512).
        # 입력 위치 축 S를 따라 '각 비중 × 각 상태'를 더한 가중합이다.
        # Bahdanau의 torch.sum(weights * H_encoder, dim=1)과 같은 종류의 계산이다.
        # 여기도 변환된 WH가 아니라 원본 H_encoder를 가중합한다.
        context_vector = torch.bmm(attention_weights.unsqueeze(1), H_encoder)  # (batch, 1, hidden_dim)
        # 가운데 크기 1인 축을 없앤다. (B, 1, H) → (B, H), 즉 (1, 512).
        # 인자를 생략한 squeeze()는 B=1일 때 배치 축까지 없앨 수 있으므로 축을 지정한다.
        context_vector = context_vector.squeeze(1)  # (batch, hidden_dim)

        # 반환 크기: context (B, H), weights (B, S).
        # Bahdanau 예제의 weights (B, S, 1)과 크기 표기는 다르지만 위치별 비중이라는 의미는 같다.
        return context_vector, attention_weights

# 설정
# 상태 벡터 길이 H를 다시 지정한다.
hidden_dim = 512
# attention 변수에 새 LuongAttention 객체를 저장한다.
# 앞의 Bahdanau 모델과 파라미터를 공유하거나 이어서 계산하지 않는다.
attention = LuongAttention(hidden_dim)

# 입력 데이터 (배치 크기 = 1)
# 인코더 상태를 흉내 낸 새 실수 난수 (1, 10, 512)를 만든다.
# 앞 셀과 같은 크기지만 새로 뽑은 값이므로 두 모델 성능을 비교하는 실험은 아니다.
enc_state = torch.rand((1, 10, hidden_dim))  # (batch, seq_len, hidden_dim)
# 비교할 디코더 상태 한 개를 (1, 512) 크기로 만든다.
dec_state = torch.rand((1, hidden_dim))  # (batch, hidden_dim)

# 실행
# Luong Attention의 점수·비중·가중합을 계산한다.
# 출력에는 context의 크기를 찍는 print가 없지만, 반환값에는 context가 포함된다.
# 반환된 context 크기는 (1, 512), weights 크기는 (1, 10)이다.
# 이 예제 역시 학습이나 실제 문장 번역은 하지 않는다.
_ = attention(enc_state, dec_state)

[ H_encoder ] Shape: torch.Size([1, 10, 512])
[ W_encoder X H_encoder ] Shape: torch.Size([1, 10, 512])
[ Score_alignment ] Shape: torch.Size([1, 10, 1])

최종 Weight:
 [[1.3472377e-02 4.3845531e-03 2.3859735e-01 1.0057037e-02 3.4070653e-01
  2.6000121e-03 9.4566040e-04 1.8204126e-01 1.0646922e-04 2.0708871e-01]]


### 두 Attention 예제 비교

| 구분 | Bahdanau 예제 | Luong 예제 |
|:--|:--|:--|
| 비교 방법 | 두 상태를 변환해 더하고 tanh 적용 | 변환된 인코더 상태와 디코더 상태의 내적 |
| 별도 중간 차원 | 100 사용 | 상태 차원 512 유지 |
| 비중 계산 | 입력 위치 축 softmax | 입력 위치 축 softmax |
| 정보를 모으는 대상 | 원본 `H_encoder` | 원본 `H_encoder` |
| 반환 context | `(1, 512)` | `(1, 512)` |
| 반환 weights | `(1, 10, 1)` | `(1, 10)` |

**두 출력의 비중을 보고 어느 모델이 더 좋다고 판단할 수는 없습니다.**
각 모델의 파라미터와 입력이 별도로 뽑은 난수이고, 학습·번역 평가를 하지 않았기 때문입니다.

## 5. 실행할 때 자주 헷갈리는 지점

**셀 재실행:** 디코더 실행 후 `hidden/cell`은 디코더의 최종 상태로 바뀝니다.
같은 연결 실습을 다시 할 때는 인코더 실행 셀부터 순서대로 실행하세요.

**GPU 출력:** 원본은 CPU 텐서를 바로 `.detach().numpy()`로 표시합니다.
모델과 데이터를 GPU로 옮겼다면 표시할 때는 `.detach().cpu().numpy()`가 필요합니다. [8]

**패딩과 학습:** 이 예제에는 패딩 위치를 제외하는 mask가 없습니다.
실제 길이가 다른 문장을 패딩해 넣는 번역 모델로 확장할 때는 유효한 위치만 참고하도록 Attention 점수를 마스킹하는 처리가 필요합니다.
현재 출력은 난수 실습의 계산 결과이지, 학습된 모델의 의미 해석이나 번역 결과가 아닙니다.

**추가 연결 작업:** 두 Attention 클래스는 `context`를 계산해 반환할 뿐입니다.
디코더 각 시점에서 이를 호출하고 해당 context를 사용하는 연결은 원본 코드에 없습니다.

## 6. 확인한 문서

코드의 실제 동작과 차원은 원본 실행문을 기준으로 해설했습니다. API 설명은 아래 문서로 확인했습니다.

[1] [PyTorch LSTM — 입출력과 hidden/cell 크기](https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.rnn.LSTM.html)  
[2] [PyTorch Embedding — 토큰 번호로 벡터 조회](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding)  
[3] [PyTorch Linear — 마지막 특징 축 변환](https://docs.pytorch.org/docs/main/generated/torch.nn.Linear.html)  
[4] [Tensor.expand — 크기 1인 축 확장과 메모리 공유](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.expand.html)  
[5] [CrossEntropyLoss — logits 입력](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)  
[6] [Softmax — 지정 축에 대한 정규화](https://docs.pytorch.org/docs/main/generated/torch.nn.functional.softmax.html)  
[7] [torch.bmm — 배치별 행렬곱](https://docs.pytorch.org/docs/stable/generated/torch.bmm.html)  
[8] [Tensor.numpy — CPU와 자동미분 관련 조건](https://docs.pytorch.org/docs/2.9/generated/torch.Tensor.numpy.html)  
[9] [PyTorch 공식 Seq2seq + Attention 튜토리얼](https://docs.pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html)  
[10] [Luong 등, Effective Approaches to Attention-based Neural Machine Translation](https://aclanthology.org/D15-1166/)

## 실행 확인

Python 3.13.5, PyTorch 2.10.0+cpu, NumPy 2.3.5의 CPU 환경에서 8개 코드 셀을 위에서 아래로 실행했습니다. 위 출력은 이 주석본을 실제 실행한 결과입니다.

주석을 제외한 코드 구조가 원본 예제 8개와 같음을 확인했으며, 동일한 난수 시드에서 출력도 일치했습니다. 별도 검증에서는 배치 크기 3, 서로 다른 입력·출력 길이, Attention 비중의 합, context 가중합 및 역전파 연결도 확인했습니다. 이 검증은 계산 동작에 대한 확인이며, 모델을 학습하거나 번역 품질을 평가한 것은 아닙니다.